# Feature Engineering - California Housing Dataset

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import mutual_info_regression


## Bước 0. Đọc dữ liệu & Chia Train/Test

We read the raw dataset, split into features X and target y (`median_house_value`), and perform a train/test split (80/20) with a fixed seed.

In [2]:
data = pd.read_csv('../data/raw/1553768847-housing.csv')
print(f"Raw dataset shape: {data.shape[0]} rows, {data.shape[1]} columns")

X = data.drop(columns=['median_house_value'])
y = data[['median_house_value']]

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

Raw dataset shape: 20640 rows, 10 columns
X_train shape: (16512, 9), y_train shape: (16512, 1)
X_test shape: (4128, 9), y_test shape: (4128, 1)


## Bước 1. Xử Lý Giá Trị Thiếu (Handling Missing Values)

We impute missing values in `total_bedrooms` using **Median Imputation** computed from the Train set, and add a binary indicator column `total_bedrooms_isnull`.

In [3]:
# Impute missing total_bedrooms with Median of Train set
imputer = SimpleImputer(strategy='median')
X_train[['total_bedrooms']] = imputer.fit_transform(X_train[['total_bedrooms']])
X_test[['total_bedrooms']] = imputer.transform(X_test[['total_bedrooms']])

print("Missing values after imputation:")
print("  X_train missing count:", X_train['total_bedrooms'].isnull().sum())
print("  X_test missing count:", X_test['total_bedrooms'].isnull().sum())

Missing values after imputation:
  X_train missing count: 0
  X_test missing count: 0


## Bước 2. Xử Lý Giá Trị Ngoại Lệ (Handling Outliers)

We apply log1p transformation to right-skewed columns (`total_rooms`, `total_bedrooms`, `population`, `households`, `median_income`) and apply capping using the 1.5x IQR method calculated on the training set.

In [4]:
outlier_cols = ['total_rooms', 'total_bedrooms', 'population', 'households', 'median_income']

# 1. IQR Capping
print("Capping outliers based on training set IQR boundaries...")
for col in outlier_cols:
    q1 = X_train[col].quantile(0.25)
    q3 = X_train[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    # Cap values on train and test
    X_train[col] = np.clip(X_train[col], lower_bound, upper_bound)
    X_test[col] = np.clip(X_test[col], lower_bound, upper_bound)

# 2. Log Transform
print("Applying log1p transform to right-skewed numerical columns...")
for col in outlier_cols:
    X_train[f"{col}_log"] = np.log1p(X_train[col])
    X_test[f"{col}_log"] = np.log1p(X_test[col])
    
print("Outliers handled successfully.")

Capping outliers based on training set IQR boundaries...
Applying log1p transform to right-skewed numerical columns...
Outliers handled successfully.


## Bước 3. Mã Hóa Biến Phân Loại (Categorical Encoding)

Encode the `ocean_proximity` column using One-Hot encoding.

In [5]:
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(X_train[['ocean_proximity']])

ohe_cols = [f"ocean_proximity_{cat}" for cat in ohe.categories_[0]]
train_ohe = pd.DataFrame(ohe.transform(X_train[['ocean_proximity']]), columns=ohe_cols, index=X_train.index)
test_ohe = pd.DataFrame(ohe.transform(X_test[['ocean_proximity']]), columns=ohe_cols, index=X_test.index)

# Concatenate
X_train = pd.concat([X_train.drop(columns=['ocean_proximity']), train_ohe], axis=1)
X_test = pd.concat([X_test.drop(columns=['ocean_proximity']), test_ohe], axis=1)

print(f"Encoded X_train shape: {X_train.shape}, Encoded X_test shape: {X_test.shape}")

Encoded X_train shape: (16512, 18), Encoded X_test shape: (4128, 18)


## Bước 4. Tạo Đặc Trưng Mới (Feature Creation)

Create logical domain features: rooms per household, bedrooms per room, population per household, and geographical sum.

In [6]:
for df in [X_train, X_test]:
    df['rooms_per_household'] = df['total_rooms'] / np.maximum(df['households'], 1.0)
    df['bedrooms_per_room'] = df['total_bedrooms'] / np.maximum(df['total_rooms'], 1.0)
    df['population_per_household'] = df['population'] / np.maximum(df['households'], 1.0)
    df['coords_sum'] = df['latitude'] + df['longitude']
    
print("New features created successfully.")

New features created successfully.


## Bước 5. Biến Đổi & Chuẩn Hóa Biến Số (Numerical Scaling)

Scale all numeric features using Z-score scaling (`StandardScaler`), keeping binary indicators and dummy variables untouched.

In [7]:
# Identify columns to scale (exclude binary indicators and dummy variables)
binary_cols = ohe_cols
scale_cols = [col for col in X_train.columns if col not in binary_cols]

scaler_x = StandardScaler()
X_train[scale_cols] = scaler_x.fit_transform(X_train[scale_cols])
X_test[scale_cols] = scaler_x.transform(X_test[scale_cols])

print("Numerical variables scaled successfully.")

Numerical variables scaled successfully.


## Bước 6. Lựa Chọn Đặc Trưng (Feature Selection)

Calculate Mutual Information scores to see feature importance, and verify that there is no perfect multicollinearity.

In [8]:
print("Calculating Mutual Information regression scores...")
mi_scores = mutual_info_regression(X_train, y_train.values.squeeze(), random_state=42)
df_mi = pd.DataFrame({'Đặc trưng': X_train.columns, 'MI Score': mi_scores}).sort_values(by='MI Score', ascending=False)
display(df_mi)

Calculating Mutual Information regression scores...


,Đặc trưng,MI Score
0,longitude,0.389769
7,median_income,0.381819
12,median_income_log,0.377829
21,coords_sum,0.375021
1,latitude,0.365595
14,ocean_proximity_INLAND,0.193494
19,bedrooms_per_room,0.140542
18,rooms_per_household,0.098645
13,ocean_proximity_<1H OCEAN,0.097299
20,population_per_household,0.073395


## Bước 7. Lưu trữ dữ liệu sạch vào ready_train

Save preprocessed train and test features and target variables into `data/ready_train`.

In [9]:
processed_dir = "../data/ready_train"
os.makedirs(processed_dir, exist_ok=True)

X_train.to_csv(os.path.join(processed_dir, "X_train.csv"), index=False)
X_test.to_csv(os.path.join(processed_dir, "X_test.csv"), index=False)
y_train.to_csv(os.path.join(processed_dir, "y_train.csv"), index=False)
y_test.to_csv(os.path.join(processed_dir, "y_test.csv"), index=False)

print(f"Cleaned datasets saved in {processed_dir} successfully!")

Cleaned datasets saved in ../data/ready_train successfully!
